# Py3plex 10-Minute Tutorial

This notebook contains the complete 10-minute tutorial for py3plex. You'll learn how to:

* Create multilayer networks from scratch
* Load networks from files
* Compute network statistics
* Detect communities
* Visualize networks

## Installation

First, install py3plex if it's not already installed:

In [ ]:
# Install py3plex in Colab or your local environment
!pip install py3plex -q

## Part 1: Create Your First Multilayer Network

Let's create a simple multilayer network with two layers.

In [ ]:
from py3plex.core import multinet

# Create a new multilayer network
network = multinet.multi_layer_network()

# Add edges within layers (this automatically creates nodes)
# Format: [source_node, source_layer, target_node, target_layer, weight]
network.add_edges([
    ['A', 'layer1', 'B', 'layer1', 1],
    ['B', 'layer1', 'C', 'layer1', 1],
    ['A', 'layer2', 'B', 'layer2', 1],
    ['B', 'layer2', 'D', 'layer2', 1]
], input_type="list")

# Display basic statistics
print("\nBasic Statistics:")
network.basic_stats()

## Part 2: Query Network with the DSL

Use py3plex's SQL-like DSL to query nodes and compute metrics.

In [ ]:
from py3plex.dsl import Q, L

# Query all nodes and compute degree
result = (
    Q.nodes()
     .from_layers(L["*"])  # All layers
     .compute("degree")
     .execute(network)
)

# Convert to pandas for easy viewing
df = result.to_pandas()
print("\nNode degrees:")
print(df)

## Part 3: Compute Advanced Metrics

Compute centrality measures to identify important nodes.

In [ ]:
# Compute multiple centrality measures
result = (
    Q.nodes()
     .from_layers(L["*"])
     .compute("degree", "betweenness_centrality", "closeness_centrality")
     .order_by("-degree")  # Sort by degree descending
     .execute(network)
)

df = result.to_pandas()
print("\nNode centralities:")
print(df)

## Part 4: Visualize the Network

Create a visualization of the multilayer network.

In [ ]:
from py3plex.visualization.multilayer import draw_multilayer_default
import matplotlib.pyplot as plt

# Create visualization
plt.figure(figsize=(10, 6))
draw_multilayer_default([network], display=True)
plt.title("Multilayer Network Visualization")
plt.show()

## Part 5: Create a Larger Network Example

Let's create a more realistic network with multiple people across social and work layers.

In [ ]:
# Create a larger network
larger_network = multinet.multi_layer_network()

# Add social connections
larger_network.add_edges([
    ['Alice', 'social', 'Bob', 'social', 1],
    ['Bob', 'social', 'Charlie', 'social', 1],
    ['Charlie', 'social', 'David', 'social', 1],
    ['Alice', 'social', 'Eve', 'social', 1],
    # Add work connections
    ['Alice', 'work', 'Bob', 'work', 1],
    ['Bob', 'work', 'David', 'work', 1],
    ['Charlie', 'work', 'Eve', 'work', 1],
], input_type="list")

larger_network.basic_stats()

## Part 5b: Use Built-in Datasets

Py3plex includes built-in datasets (similar to scikit-learn) for quick experimentation.

In [ ]:
from py3plex.datasets import load_aarhus_cs, make_random_multilayer, list_datasets

# List available datasets
print("Available datasets:")
print(list_datasets())

# Create a random multilayer network for quick testing
print("\nCreating a random multilayer network...")
random_net = make_random_multilayer(n_nodes=20, n_layers=3, p_intra=0.2, p_inter=0.1)
random_net.basic_stats()

# You can also load real-world datasets like:
# aarhus_net = load_aarhus_cs()  # Computer science social network
# aarhus_net.basic_stats()

## Part 5c: Dplyr-Style Graph Operations

Use chainable operations to filter, transform, and analyze nodes.

In [ ]:
from py3plex.graph_ops import nodes
import numpy as np

# Dplyr-style API: Chain operations for elegant data manipulation
# This is similar to R's dplyr or pandas method chaining
df = (
    nodes(random_net)  # Start with all nodes
    .filter(lambda n: n["degree"] > 1)  # Keep only connected nodes
    .mutate(degree_squared=lambda n: n["degree"] ** 2)  # Create new column
    .mutate(importance=lambda n: n["degree"] / n["degree_squared"])  # Compute importance score
    .arrange("degree", reverse=True)  # Sort by degree (highest first)
    .to_pandas()  # Convert to pandas DataFrame
)

print("\nNodes with degree > 1 (sorted by degree):")
print(df.head(10))

# Group by layer and compute aggregate statistics
# Similar to SQL GROUP BY or pandas groupby
layer_stats = (
    nodes(random_net)
    .group_by("layer")  # Group nodes by their layer
    .summarise(
        avg_degree=("degree", np.mean),  # Average degree per layer
        max_degree=("degree", np.max),   # Maximum degree per layer
        node_count=("degree", len)        # Count nodes per layer
    )
    .to_pandas()
)

print("\nLayer-wise statistics:")
print(layer_stats)

## Part 7: Detect Communities

Use community detection to find groups of densely connected nodes.

In [ ]:
from py3plex.algorithms.community_detection.multilayer_modularity import louvain_multilayer
from collections import Counter

# Run multilayer community detection
partition_dict, modularity = louvain_multilayer(larger_network)

print(f"\nModularity: {modularity:.3f}")
print(f"Number of communities: {len(set(partition_dict.values()))}")

# Show community assignments
print("\nCommunity assignments:")
for node, community in sorted(partition_dict.items())[:10]:
    print(f"  {node} → Community {community}")

## Part 7b: Centrality Metrics with Uncertainty

Compute multiple centrality measures with confidence intervals using bootstrap.

In [ ]:
from py3plex.dsl import Q, L

# Compute multiple centrality metrics with uncertainty quantification
# This helps understand the reliability of centrality rankings
result = (
    Q.nodes()
     .from_layers(L["*"])  # All layers
     .uq(method="bootstrap", n_samples=30, ci=0.95)  # Bootstrap with 95% CI
     .compute(
         "degree",
         "betweenness_centrality",
         "closeness_centrality",
         "eigenvector_centrality"
     )
     .order_by("-betweenness_centrality")  # Sort by betweenness (mean)
     .limit(10)  # Top 10 nodes
     .execute(larger_network)
)

# Convert to DataFrame with expanded uncertainty columns
df = result.to_pandas(expand_uncertainty=True)

print("\nTop 10 nodes with centrality uncertainty:")
print("\nBetweenness Centrality (with 95% CI):")
print(df[['id', 'betweenness_centrality', 'betweenness_centrality_std',
          'betweenness_centrality_ci95_low', 'betweenness_centrality_ci95_high']].head())

print("\nDegree Centrality (with 95% CI):")
print(df[['id', 'degree', 'degree_std', 'degree_ci95_low', 'degree_ci95_high']].head())

# Identify nodes with stable vs unstable centrality
print("\nCentrality Stability Analysis:")
df['betweenness_cv'] = df['betweenness_centrality_std'] / df['betweenness_centrality']  # Coefficient of variation
stable_nodes = df[df['betweenness_cv'] < 0.2]  # CV < 20% is stable
print(f"Stable nodes (low uncertainty): {len(stable_nodes)}")
print(f"Unstable nodes (high uncertainty): {len(df) - len(stable_nodes)}")

## Part 8: Sklearn-Style Pipeline

Use composable pipelines to chain operations (similar to scikit-learn).

In [ ]:
from py3plex.pipeline import Pipeline, ComputeStats, FilterNodes, LouvainCommunity

# Create a sklearn-style pipeline to process networks
# Pipelines allow you to chain multiple transformations
pipeline = Pipeline([
    # Step 1: Compute network statistics (degree, betweenness)
    ("compute_stats", ComputeStats(metrics=["degree", "betweenness_centrality"])),
    
    # Step 2: Filter out low-degree nodes (noise reduction)
    ("filter_low_degree", FilterNodes(condition=lambda n: n.get("degree", 0) > 1)),
    
    # Step 3: Detect communities using Louvain algorithm
    ("detect_communities", LouvainCommunity(resolution=1.0)),
])

# Apply the entire pipeline to the network
# This runs all steps in sequence
result_network = pipeline.fit_transform(random_net)

print("\nPipeline executed successfully!")
print(f"Filtered network has {len(result_network.get_nodes())} nodes")

# Access computed metrics (added by pipeline steps)
print("\nSample node attributes after pipeline:")
for node in list(result_network.get_nodes())[:3]:
    attrs = result_network.get_node_attributes(node)
    print(f"  {node}: degree={attrs.get('degree', 'N/A')}, "
          f"community={attrs.get('community', 'N/A')}")

## Summary

In this tutorial, you learned:

1. ✅ How to create multilayer networks from scratch
2. ✅ How to query networks using the DSL
3. ✅ How to compute centrality measures
4. ✅ How to visualize multilayer networks
5. ✅ How to use built-in datasets
6. ✅ How to use dplyr-style graph operations
7. ✅ How to detect communities
8. ✅ **How to compute centrality with uncertainty (NEW)**
9. ✅ How to use sklearn-style pipelines

## Next Steps

* Explore the [full documentation](https://skblaz.github.io/py3plex/)
* Try the [DSL tutorial](query_with_dsl.ipynb)
* Learn about [dynamics simulation](simulate_dynamics.ipynb)
* Check out [example scripts](https://github.com/SkBlaz/py3plex/tree/main/examples)
* Dive deeper into [community detection](community_detection.ipynb)